# Chapter 1 — Star or galaxy? Classifying individual objects

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/01_star_galaxy_objects.ipynb)

**Goal.** A satellite search counts *stars*. At the faint magnitudes where the newest satellites live, faint
galaxies outnumber halo stars, and many of them look like points. In this chapter we measure how well four
surveys separate stars from galaxies, using as the truth the Hubble Space Telescope (HST) images of the COSMOS
field, which resolve galaxies that ground-based images cannot.

**You will learn**
1. why a point-spread-function (PSF) magnitude and an extended-model magnitude differ for galaxies, and why
   this difference vanishes for faint, compact galaxies (a toy image model);
2. how SDSS, Pan-STARRS1 (PS1), the Legacy Surveys (LS) and Hyper Suprime-Cam (HSC) classify sources;
3. to crossmatch catalogues and measure **completeness** (fraction of true stars kept), **leakage** (fraction of
   true galaxies classified as stars) and **purity** (fraction of "stars" that are stars) with binomial errors;
4. that purity depends on the ratio of stars to galaxies, which changes with magnitude and sky position;
5. where the leaking galaxies fall in colour–colour space.

Two errors matter differently: a galaxy called a star adds contamination and can create a false detection; a star
called a galaxy loses signal and costs completeness.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats                               # beta distribution, for binomial (Jeffreys) intervals
from scipy.spatial import cKDTree                     # k-d tree, for nearest-neighbour crossmatching

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)

## 1.1 Why faint galaxies look like stars: a toy image

An image of a source is its true light distribution convolved with the PSF (atmosphere + telescope), plus noise.
A star is a delta function, so its image *is* the PSF. A galaxy is extended, so its image is broader — but only
slightly, if the galaxy is much smaller than the PSF. To decide, we fit two models to each image: the PSF, and a
slightly extended model (here an exponential disc of half-light radius 0.25″, convolved with the PSF), and compare
their $\chi^2$. With little signal the two fits become equally good, and the classification turns into a coin toss.

Below, the PSF is a Gaussian with FWHM 1.3″ (typical ground-based seeing) on 0.4″ pixels with uniform noise. We set
the threshold on $\Delta\chi^2 = \chi^2_{\rm PSF} - \chi^2_{\rm ext}$ so that 5% of simulated *stars* are called
extended, and measure how many *galaxies* are called stars as a function of S/N.

In [ ]:
rng = np.random.default_rng(1)                                # fixed seed: the simulated images are reproducible
pix, n = 0.4, 25                                              # arcsec per pixel, image size in pixels
# The 25 x 0.4″ = 10″ stamp is ~8 PSF FWHM wide, so both profiles fall to ~0 at its edges (the FFT
# convolution below is circular and would otherwise wrap light around).
x = (np.arange(n) - n // 2) * pix                             # pixel-centre offsets [arcsec], 0 at the central pixel
X, Y = np.meshgrid(x, x)
sig_psf = 1.3 / 2.355                                         # Gaussian sigma [arcsec]: FWHM 1.3″ = 2.355 sigma
psf = np.exp(-(X**2 + Y**2) / (2 * sig_psf**2)); psf /= psf.sum()   # star image, normalised to unit flux
re = 0.25                                                     # galaxy half-light radius, arcsec
gal = np.exp(-1.678 * np.hypot(X, Y) / re)                    # exponential profile, r_e = 1.678 r_s
# Convolve galaxy and PSF with FFTs: ifftshift moves each profile's centre to pixel (0, 0), as the FFT expects,
# and fftshift moves the result back to the stamp centre.
ext = np.real(np.fft.ifft2(np.fft.fft2(np.fft.ifftshift(gal)) * np.fft.fft2(np.fft.ifftshift(psf))))
ext = np.fft.fftshift(ext); ext /= ext.sum()                  # galaxy convolved with the PSF, unit flux


def delta_chi2(img, noise):
    """chi2(PSF fit) - chi2(extended fit); each fit has one free parameter, the flux (linear least squares).

    img has shape (..., n, n): one image or a stack of images. The result has shape img.shape[:-2] and is
    positive when the extended model fits better.
    """
    out = []
    for t in (psf, ext):
        # least-squares flux of template t: f = sum(img * t) / sum(t^2), per image
        f = (img * t).sum(axis=(-2, -1)) / (t * t).sum()
        out.append((((img - f[..., None, None] * t) / noise)**2).sum(axis=(-2, -1)))
    return out[0] - out[1]


noise = 1.0                                                   # per-pixel Gaussian noise sigma (arbitrary flux units)
snr = np.array([3, 5, 7, 10, 15, 20, 30, 50, 100])            # from near the detection limit to bright sources
leak = []
for s in snr:
    # The variance of the optimal PSF-weighted flux is noise^2 / sum(psf^2); this sets the flux that gives S/N = s.
    flux = s * noise * np.sqrt(1 / (psf**2).sum())            # S/N of an optimal PSF-weighted flux measurement
    # 4000 noisy images of each kind, shape (4000, 25, 25): binomial error of a fraction <= 0.5 / sqrt(4000) = 0.008
    stars = flux * psf + rng.normal(0, noise, (4000, n, n))
    gals = flux * ext + rng.normal(0, noise, (4000, n, n))
    thr = np.percentile(delta_chi2(stars, noise), 95)          # 5% of stars are called extended
    leak.append(np.mean(delta_chi2(gals, noise) < thr))         # galaxies called stars
leak = np.array(leak)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
# left: cut through the centre of each model, normalised to its peak
ax[0].plot(x, psf[n // 2] / psf.max(), label='star (PSF)')
ax[0].plot(x, ext[n // 2] / ext.max(), label='galaxy ⊗ PSF')
ax[0].set(xlabel='arcsec', ylabel='normalised profile', title='the two models, FWHM 1.3″, galaxy r_e = 0.25″')
ax[0].legend()
# middle: one noisy galaxy image; `flux` is left over from the last loop step, S/N = snr[-1] = 100
ax[1].imshow(flux * ext + rng.normal(0, noise, (n, n)), cmap='Greys', origin='lower')
ax[1].set(title=f'a simulated galaxy at S/N = {snr[-1]}', xticks=[], yticks=[])
# right: leakage (galaxies called stars) against S/N
ax[2].semilogx(snr, leak, 'o-')
ax[2].set(xlabel='S/N', ylabel='fraction of galaxies classified as stars', ylim=(0, 1),
          title='5% of stars called extended')
plt.tight_layout()
print('fraction of galaxies called stars at S/N', dict(zip(snr.tolist(), leak.round(2).tolist())))
# Test (STRUCTURE.md: misclassification rises as S/N falls). With no shape information the leakage tends to 0.95
# (galaxies pass the cut as often as stars do); the 0.4 margin is ~50 times the binomial error of 0.008.
# Measured: 0.94 at S/N = 3, 0.00 at S/N = 100.
assert leak[0] > leak[-1] + 0.4          # at S/N = 3 most such galaxies pass as stars, at S/N = 100 few do

**Figure 1.1.** A toy model of star–galaxy separation: simulated 25 × 25-pixel images (0.4″ pixels), a Gaussian PSF
of FWHM 1.3″ and uniform Gaussian pixel noise. Left: cuts through the centre, normalised to the peak, of the PSF
(blue: a star) and of an exponential galaxy of half-light radius $r_e = 0.25″$ convolved with the PSF (orange) — the
compact galaxy is only slightly broader than a star. Middle: one simulated image of that galaxy at S/N = 100 (grey
scale, darker is brighter). Right: the fraction of 4000 simulated galaxies classified as stars, against S/N (log
axis, S/N = 3–100), when the threshold on $\Delta\chi^2 = \chi^2_{\rm PSF} - \chi^2_{\rm ext}$ is set so that 5%
of 4000 simulated stars are called extended. The leakage is 0.00 at S/N = 100, 0.23 at 50, 0.61 at 30, 0.89 at 10
and 0.94 at 3, close to the 0.95 reached when the images carry no shape information. Real galaxies have a range of
sizes, so real leakage is a mixture of such curves; but the lesson holds: **the absence of measurable extent at low
S/N is not evidence that a source is a star.**

## 1.2 The truth: COSMOS seen by HST

Leauthaud et al. (2007, https://ui.adsabs.harvard.edu/abs/2007ApJS..172..219L) classified every source in the
HST/ACS images of COSMOS (resolution ~0.1″) using its peak surface brightness and magnitude: `mu_class` = 1
galaxy, 2 star, 3 artifact. At these resolutions almost every galaxy is resolved, so we take `mu_class` as the
truth (it is not perfect at the faintest magnitudes, but far better than any ground-based survey).

We crossmatch each ground-based survey to HST: for every survey source, the nearest HST source within 1″.
Survey sources outside the HST footprint, or without an HST counterpart, are dropped.

In [ ]:
hst = load('cosmos_hst')                                      # COSMOS HST/ACS catalogue, mag_auto < 25.5, 0.6° radius
hst = hst[hst.mu_class.isin([1, 2])].reset_index(drop=True)   # keep galaxies (1) and stars (2), drop artifacts (3)
print(f'HST sources: {len(hst):,} ({(hst.mu_class == 2).sum():,} stars)')


def xmatch(ra, dec, ra2, dec2, radius_arcsec=1.0):
    """Index of the nearest (ra2, dec2) neighbour within radius for every (ra, dec); -1 if none (flat-sky, small field).

    RA differences are scaled by cos(median dec), which is accurate over the 0.6° field. Angles in deg, radius
    in arcsec.
    """
    c = np.cos(np.radians(np.median(dec)))
    tree = cKDTree(np.column_stack([ra2 * c, dec2]))
    # with distance_upper_bound, sources without a neighbour get d = inf (and an out-of-range index)
    d, i = tree.query(np.column_stack([ra * c, dec]), distance_upper_bound=radius_arcsec / 3600)
    return np.where(np.isfinite(d), i, -1)


def lsmag(flux):
    """Legacy Surveys fluxes are in nanomaggies: m = 22.5 - 2.5 log10(flux)."""
    return 22.5 - 2.5 * np.log10(np.where(flux > 0, flux, np.nan))   # non-positive fluxes -> NaN magnitude


# For each survey: the catalogue, its r-band PSF magnitude, and its own star/galaxy decision.
# SDSS: type 6 = point source. PS1: r_PSF - r_Kron < 0.05 mag. LS: Tractor type 'PSF' (flux_r is the Tractor model
# flux, i.e. the PSF flux for type 'PSF'). HSC: i_extendedness_value is 0 (point) or 1 (extended); < 0.5 picks 0
# and rejects NaN. All files: 0.6° around COSMOS (RA, Dec) = (150.12°, 2.21°).
surveys = {}
d = load('cosmos_sdss'); surveys['SDSS'] = (d, d.psfmag_r.to_numpy(), (d.type == 6).to_numpy())
d = load('cosmos_ps1'); surveys['PS1'] = (d, d.rpsfmag.to_numpy(), (d.rpsfmag - d.rkronmag < 0.05).to_numpy())
d = load('cosmos_ls'); surveys['LS'] = (d, lsmag(d.flux_r.to_numpy()), (d.type == 'PSF').to_numpy())
d = load('cosmos_hsc'); surveys['HSC'] = (d, d.r_psfflux_mag.to_numpy(), (d.i_extendedness_value < 0.5).to_numpy())
# truth[k] = (is an HST star, has an HST match within 1″), boolean arrays aligned with survey k's catalogue.
# Unmatched sources get is_star = False and are excluded later through `matched`; np.maximum(i, 0) only keeps
# the index valid for them.
truth = {}
for k, (d, m, cl) in surveys.items():
    i = xmatch(d.ra.to_numpy(), d.dec.to_numpy(), hst.ra.to_numpy(), hst.dec.to_numpy())
    truth[k] = np.where(i >= 0, hst.mu_class.to_numpy()[np.maximum(i, 0)] == 2, False), i >= 0
    print(f'{k:5s}: {len(d):6,} sources, {np.mean(i >= 0):.2f} matched to HST within 1″')

The matched fraction is high for SDSS, LS and HSC; PS1 has more faint, spurious or blended stack detections
without an HST counterpart. How each survey decides "star":

| Survey | Classifier | Star if |
|---|---|---|
| SDSS | `type`, from PSF − cModel magnitude (Stoughton et al. 2002) | `type = 6`, i.e. $m_{\rm PSF} - m_{\rm cModel} < 0.145$ in the summed bands |
| PS1 | PSF − Kron magnitude (Farrow et al. 2014) | $r_{\rm PSF} - r_{\rm Kron} < 0.05$ |
| LS DR9 | Tractor model selection (Lang et al. 2016; Dey et al. 2019) | `type = 'PSF'`: a point source fits as well as a galaxy model |
| HSC PDR3 | `extendedness` from PSF − cModel flux (Bosch et al. 2018) | `i_extendedness_value = 0` |

## 1.3 The magnitude difference behind the SDSS classifier

For a point source the PSF and model magnitudes agree; for an extended source the PSF fit misses light in the
wings, so $\Delta m = m_{\rm PSF} - m_{\rm cModel} > 0$. The ratio of model to PSF flux is $10^{0.4 \Delta m}$.

In [ ]:
d, m, cl = surveys['SDSS']
is_star, matched = truth['SDSS']
dm = (d.psfmag_r - d.cmodelmag_r).to_numpy()                  # Delta m [mag]: > 0 when the PSF fit misses light
fig, ax = plt.subplots(figsize=(7, 4.5))
# all SDSS sources (any type) with an HST match; galaxies drawn first, stars on top
ax.scatter(m[matched & ~is_star], dm[matched & ~is_star], s=1, c='C1', alpha=0.4, label='HST galaxies')
ax.scatter(m[matched & is_star], dm[matched & is_star], s=1, c='C0', alpha=0.6, label='HST stars')
ax.axhline(0.145, c='k', ls='--', lw=1, label='SDSS threshold 0.145')
ax.set(xlim=(16, 24), ylim=(-0.5, 2.5), xlabel='SDSS $r_{\\rm PSF}$', ylabel='$r_{\\rm PSF} - r_{\\rm cModel}$')
ax.legend(markerscale=6)
plt.tight_layout()

**Figure 1.2.** SDSS DR9 sources of all types in COSMOS (`cosmos_sdss`) with an HST/ACS counterpart within 1″:
$\Delta m = r_{\rm PSF} - r_{\rm cModel}$ against $r_{\rm PSF}$ (observed SDSS r-band magnitudes) for HST stars
(blue, `mu_class` = 2) and HST galaxies (orange, `mu_class` = 1); the dashed line is the SDSS threshold Δm = 0.145.
The stellar sequence (blue) sits at $\Delta m \approx 0$ and widens as the errors grow beyond r ≈ 20; galaxies
(orange) have $\Delta m > 0$, up to 2.5 and more, at bright magnitudes but converge onto the stellar sequence towards
$r \approx 22$, where the two populations can no longer be separated by shape. (SDSS uses the sum of all bands, so
the r-band threshold drawn here is only indicative.)

## 1.4 Completeness, leakage and purity versus magnitude

In each magnitude bin, with $N_\star$ true stars and $N_G$ true galaxies:

- completeness $C_\star = P(\text{called star} \mid \text{star})$,
- leakage $L_G = P(\text{called star} \mid \text{galaxy})$,
- purity $= N_\star C_\star / (N_\star C_\star + N_G L_G)$.

Completeness and leakage are properties of the classifier; purity also depends on $N_G / N_\star$. We give 68%
binomial intervals (Jeffreys: a Beta(k + ½, n − k + ½) posterior for k successes in n trials) and skip bins with
fewer than 30 objects of the relevant class.

In [ ]:
def rate(k, n):
    """Fraction with a 68% Jeffreys interval; NaN if n < 30.

    k successes in n trials; the interval is the 16th-84th percentile range of the Beta(k + 1/2, n - k + 1/2)
    posterior. Returns (k / n, lower, upper).
    """
    if n < 30:                                    # too few objects of this class in the bin: skip it
        return np.nan, np.nan, np.nan
    lo, hi = stats.beta.ppf([0.16, 0.84], k + 0.5, n - k + 0.5)
    return k / n, lo, hi


edges = np.arange(17, 24.01, 0.5)                 # 0.5-mag bins in each survey's own r magnitude, 17 to 24
mid = 0.5 * (edges[1:] + edges[:-1])
# res[k]: shape (14 bins, 9) = (completeness, lo, hi, leakage, lo, hi, purity, lo, hi) per bin, HST-matched only
res = {}
for k, (d, m, cl) in surveys.items():
    is_star, matched = truth[k]
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        b = matched & (m >= lo) & (m < hi)
        s, g = b & is_star, b & ~is_star          # true stars and true galaxies in this bin
        # completeness = called star among true stars; leakage = called star among true galaxies;
        # purity = true stars among the objects called stars
        rows.append(rate(cl[s].sum(), s.sum()) + rate(cl[g].sum(), g.sum()) + rate(is_star[b & cl].sum(), (b & cl).sum()))
    res[k] = np.array(rows)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
# col: column of res holding the fraction; col + 1 and col + 2 hold its interval
for j, (title, col) in enumerate([('completeness  P(star | HST star)', 0), ('leakage  P(star | HST galaxy)', 3),
                                  ('purity of the "star" sample', 6)]):
    for k in res:
        r = res[k]
        err = np.clip([r[:, col] - r[:, col + 1], r[:, col + 2] - r[:, col]], 0, None)   # k = n: interval below 1
        ax[j].errorbar(mid, r[:, col], yerr=err, marker='o', ms=3, label=k)
    ax[j].set(xlabel='$r_{\\rm PSF}$ (LS: Tractor model $r$)', title=title, ylim=(0, 1.02))
ax[0].legend()
plt.tight_layout()


def binval(k, col, lo, hi):
    """Pooled fraction over magnitude bins [lo, hi) for survey k (recomputed from counts, not averaged).

    col = 'completeness' returns P(called star | HST star); anything else returns the contamination of the
    "star" sample, 1 - purity. HST-matched sources only.
    """
    d, m, cl = surveys[k]
    is_star, matched = truth[k]
    b = matched & (m >= lo) & (m < hi)
    if col == 'completeness':
        return cl[b & is_star].mean()
    return 1 - is_star[b & cl].mean()            # contamination = 1 - purity


print('SDSS completeness r < 20:', round(binval('SDSS', 'completeness', 17, 20), 3))
for k in surveys:
    print(f'{k:5s} contamination of the "star" sample at 21.5 < r < 22.5: {binval(k, "contamination", 21.5, 22.5):.2f}')
# Tests (STRUCTURE.md), with measured values:
# SDSS keeps > 90% of stars at 17 < r < 20 (0.988);
assert binval('SDSS', 'completeness', 17, 20) > 0.9
# more than 20% of SDSS "stars" at 21.5 < r < 22.5 are galaxies (0.35), where a satellite's turn-off lies;
assert binval('SDSS', 'contamination', 21.5, 22.5) > 0.2
# HSC is less contaminated than SDSS in the same range (0.01 vs 0.35).
assert binval('HSC', 'contamination', 21.5, 22.5) < binval('SDSS', 'contamination', 21.5, 22.5)

**Figure 1.3.** Star–galaxy classification in COSMOS against HST truth, for sources with an HST match within 1″, in
0.5-mag bins of each survey's own r magnitude (17–24; PSF magnitudes, except LS, where it is the Tractor model flux):
SDSS `type = 6` (blue), PS1 $r_{\rm PSF} - r_{\rm Kron} < 0.05$ (orange), LS `type = 'PSF'` (green) and HSC
`i_extendedness_value = 0` (red). Error bars are 68% Jeffreys binomial intervals; bins with fewer than 30 objects
of the relevant class are omitted. Left: completeness, P(called star | HST star). SDSS and LS keep nearly all stars to
r ≈ 21; PS1 loses ~10% even at bright magnitudes with this simple cut; HSC loses *bright* stars (completeness ≈ 0 at
r < 18, rising to 0.9 only at r ≈ 21): they saturate in the 8-m Subaru images, and `extendedness` is then
meaningless — a reminder that every classifier has a valid magnitude range. Middle: leakage, P(called star | HST
galaxy). It rises steeply beyond r ≈ 21 for SDSS and PS1 (to ≈ 0.3 at r ≈ 23.5), as in the toy model; LS leaks
mainly beyond r ≈ 23, HSC hardly at all. Right: purity, the fraction of "stars" confirmed by HST. At r ≈ 22, where the
main-sequence turn-off of a satellite at 60 kpc lies, half of the SDSS "stars" in this field are galaxies; LS
(deeper DECam images, Tractor) and especially HSC (0.6″ seeing, much deeper) do far better. Pooled over
21.5 < r < 22.5, the contamination (1 − purity) is 0.35 for SDSS, 0.52 for PS1, 0.10 for LS and 0.01 for HSC; the SDSS
completeness at r < 20 is 0.988.

## 1.5 Purity depends on where you look

Leakage is a property of the classifier, but purity is not: it also depends on the ratio $N_G/N_\star$. COSMOS is at
Galactic latitude $b = 42°$. At the same magnitude, a field at lower latitude has more stars, the same number of
galaxies, and so a purer star sample. With completeness and leakage fixed at their COSMOS values, purity for
a field with $k$ times more stars is $k N_\star C_\star / (k N_\star C_\star + N_G L_G)$:

In [ ]:
d, m, cl = surveys['SDSS']
is_star, matched = truth['SDSS']
b = matched & (m >= 21.5) & (m < 22.5)                       # the faint bin of the tests above
Ns, Ng = (b & is_star).sum(), (b & ~is_star).sum()           # HST stars and HST galaxies in the bin
C, L = cl[b & is_star].mean(), cl[b & ~is_star].mean()       # SDSS completeness and leakage, held fixed
k = np.logspace(-0.5, 1.5, 50)                               # star density relative to COSMOS: 0.3 to 30 times
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(k, k * Ns * C / (k * Ns * C + Ng * L))           # purity = k Ns C / (k Ns C + Ng L)
ax.axvline(1, c='k', lw=0.8, ls=':')                         # COSMOS itself
ax.set(xlabel='star density relative to COSMOS', ylabel='purity of SDSS "stars", 21.5 < r < 22.5', ylim=(0, 1))
plt.tight_layout()
print(f'COSMOS: N_star = {Ns}, N_gal = {Ng}, C = {C:.2f}, L = {L:.3f}')

**Figure 1.4.** Purity of SDSS "stars" with 21.5 < r < 22.5 predicted for a field with k times the COSMOS star
density (x axis, log scale, k = 0.3–30), with the galaxy density and the COSMOS completeness C = 0.77 and leakage
L = 0.110 held fixed (measured from N★ = 1813 HST stars and N_G = 6778 HST galaxies in that magnitude range):
purity = k N★C / (k N★C + N_G L). The dotted line marks COSMOS itself (k = 1). The same classifier gives a 65% pure
star sample at COSMOS, and > 90% purity in a field with ten times more stars (closer to the Galactic plane or the
bulge); in a field with a third of the COSMOS star density, purity falls to ≈ 37%. For halo searches at high latitude,
contamination is at its worst.

## 1.6 Where do the leaking galaxies fall in colour?

Stars of different temperatures form a narrow **stellar locus** in colour–colour space (Covey et al. 2007,
https://ui.adsabs.harvard.edu/abs/2007AJ....134.2398C). Galaxies at different redshifts spread around it. Colours
therefore carry information independent of shape — which we will use in Chapter 3 to decide whether an
overdensity is made of stars.

In [ ]:
d, m, cl = surveys['SDSS']
is_star, matched = truth['SDSS']
# observed SDSS PSF colours (not dereddened)
gr = (d.psfmag_g - d.psfmag_r).to_numpy()
ri = (d.psfmag_r - d.psfmag_i).to_numpy()
faint = matched & cl & (m > 21) & (m < 22.5)                 # faint SDSS "stars" (type 6) with an HST match
bright = matched & is_star & cl & (m < 20)                   # bright HST stars that SDSS also calls stars
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True, sharey=True)
ax[0].scatter(gr[bright], ri[bright], s=3, c='C0', label='HST stars, r < 20')
ax[0].set(title='bright stars: the stellar locus')
# right: the faint "stars" split by their HST class
ax[1].scatter(gr[faint & is_star], ri[faint & is_star], s=4, c='C0', label='HST stars')
ax[1].scatter(gr[faint & ~is_star], ri[faint & ~is_star], s=4, c='C1', label='HST galaxies')
ax[1].set(title='SDSS "stars" with 21 < r < 22.5')
for a in ax:
    a.set(xlim=(-0.5, 2.0), ylim=(-0.5, 2.0), xlabel='$g-r$', ylabel='$r-i$')
    a.legend(markerscale=3)
plt.tight_layout()

**Figure 1.5.** Colour–colour diagrams, $r-i$ against $g-r$ (observed SDSS PSF colours, not dereddened), of COSMOS
sources with an HST match. Left: bright stars (HST stars that SDSS also calls stars, r < 20) trace the stellar locus:
a diagonal branch for F–K stars and a vertical branch at g − r ≈ 1.4 for M dwarfs. Right: faint SDSS "stars"
(`type = 6`, 21 < r < 22.5), split by HST class. The true stars (blue) scatter more, because the errors are larger;
the galaxies (orange) concentrate at g − r ≈ 0.3–1.5 and *below* the M-dwarf branch. Colours help, but at these
errors the two distributions overlap; one object's colours cannot prove it is a star. A whole *population* of excess
objects is another matter (Chapter 3).

## Summary

- Low S/N erases the difference between a point source and a compact galaxy; faint "stars" are contaminated.
- Against HST truth: SDSS keeps > 90% of stars to r ≈ 21, but at 21.5 < r < 22.5 a large fraction of its
  "stars" in COSMOS are galaxies; LS and HSC do much better; HSC fails for bright, saturated stars.
- Completeness and leakage describe a classifier; purity also depends on the star/galaxy ratio of the field.
- Leaking galaxies occupy different colours from stars on average, but overlap them object by object.